In [ ]:
repository_filter: list[str] = []
top_n_repos: int = 30

In [ ]:
import pandas as pd
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import warnings

warnings.simplefilter("ignore")

df = dt.read_csv("../samples/test_gaps.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)

In [ ]:
if len(df) == 0:
    fig = qu.empty_figure()
else:
    bins = [0, 25, 50, 75, 100]
    labels = ["Low", "Medium", "High", "Critical"]
    df["riskBucket"] = pd.cut(
        df["riskScore"], bins=bins, labels=labels, include_lowest=True
    )

    repo_totals = df.groupby("repoShort").size().nlargest(top_n_repos)
    top_repos = repo_totals.index.tolist()
    filtered = df[df["repoShort"].isin(top_repos)]

    pivot = (
        filtered.groupby(["repoShort", "riskBucket"], observed=False)
        .size()
        .unstack(fill_value=0)
    )
    pivot = pivot.reindex(columns=labels, fill_value=0)
    pivot = pivot.loc[repo_totals.index]

    text_vals = [[str(v) for v in row] for row in pivot.values]

    fig = go.Figure(
        data=go.Heatmap(
            z=pivot.values,
            x=pivot.columns.tolist(),
            y=pivot.index.tolist(),
            colorscale=[
                [0, "#E8F5E9"],
                [0.33, "#FFE082"],
                [0.66, "#FF8A65"],
                [1, "#EF5350"],
            ],
            text=text_vals,
            texttemplate="%{text}",
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Risk: %{x}<br>"
                "Count: %{text}"
                "<extra></extra>"
            ),
            colorbar=dict(title="Gap Count"),
        )
    )
    fig.update_layout(
        title=f"Test Gap Risk Heatmap (top {len(pivot)} repositories)",
        xaxis_title="Risk Level",
        yaxis_title="",
        height=max(500, len(pivot) * 28 + 150),
        margin=dict(l=200, r=50, t=60, b=60),
        yaxis=dict(autorange="reversed"),
    )
fig.show()